## Setup

*You must run the cells in this section each time you connect to a new runtime. For example, when you return to the notebook after an idle timeout, when the runtime crashes, or when you restart or factory reset the runtime.*

Install requirements:

In [ ]:
! pip install --upgrade pip > pip.log
! pip install --upgrade ocdskingfishercolab psycopg2-binary >> pip.log

In [ ]:
# @title Import packages and load extensions { display-mode: "form" }

import gzip
import json
import os
import shutil
import tempfile
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from dateutil.relativedelta import relativedelta
from google.colab.data_table import DataTable
from google.colab.files import download
from ipywidgets import widgets
from ocdskingfishercolab import (
    authenticate_gspread,
    calculate_coverage,
    check_usability_indicators,
    download_dataframe_as_csv,
    format_coverage,
    format_thousands,
    get_publication_select_box,
    get_publications,
    indicator_checks,
    load_indicators,
    most_common_fields_to_calculate_indicators,
    plot_objects_per_stage,
    plot_objects_per_year,
    plot_release_count,
    plot_releases_by_month,
    plot_top_buyers,
    plot_usability_indicators,
    render_json,
    save_dataframe_to_sheet,
    save_dataframe_to_spreadsheet,
    set_dark_mode,
    set_light_mode,
)

# Load https://pypi.org/project/ipython-sql/
%load_ext sql
# Load https://colab.research.google.com/notebooks/data_table.ipynb
%load_ext google.colab.data_table

In [ ]:
# @title Configure the notebook environment { display-mode: "form" }

# Increase max columns so that Pandas DataFrames with many columns are rendered as data tables.
DataTable.max_columns = 50
# Remove the index from data tables for easier copy-pasting to Google Docs.
DataTable.include_index = False

# Return Pandas DataFrames instead of regular result sets.
%config SqlMagic.autopandas = True
# Don't print number of rows affected.
%config SqlMagic.feedback = False

# If you set Tools > Settings > Site > Theme to dark, uncomment this line.
# set_dark_mode()
# If you are creating plots to copy-paste into reports, uncomment this line.
# set_light_mode()

## Check the MVP status of the Data registry publications

Use this notebook to check which publications in the Data Registry pass the MVP Relevant and Active criteria, for example, for updating the MEL1 tracker upon OCP Rapid Reflection meetings.

In [ ]:
# @title Get all the publications from the registry { display-mode: "form" }

publications = get_publications()

### Check for non-frozen publications whose latest data has not been updated in the previous four calendar quarters

From the list, check also the "last_retrieved" and "update_frequency" columns. If the data is not being retrieved, check the publication log in the Data Registry to check if there is a problem with either a job or the source data itself.

In [ ]:
non_frozen_publications = [item for item in publications if not item["frozen"] and item["date_to"]]
today = datetime.now(tz=timezone.utc)
past_year = today - relativedelta(years=1)
lapsed_publications = [
    item
    for item in non_frozen_publications
    if datetime.strptime(item["date_to"], "%Y-%m-%d").astimezone(timezone.utc) < past_year
]
lapsed_publications_table = pd.DataFrame(lapsed_publications)
lapsed_publications_table

### Check non-relevant publications

Check which active publications pass and not pass the "Relevant" criterion.

In [ ]:
results = []
active_publications = [item for item in non_frozen_publications if item not in lapsed_publications]
for publication in active_publications:
    field_table = format_coverage(publication.get("coverage", {}))
    fields_list = field_table.iloc[:, 0].tolist()
    relevant, relevant_table = is_relevant(fields_list)
    relevant_table["publisher"] = publication["label"]
    relevant_table["relevant"] = relevant
    results.append(relevant_table)

Filter the non-relevant ones

In [ ]:
result = pd.concat(results)
not_relevant_publishers = result[~result["relevant"]]
non_relevant_rules = (
    not_relevant_publishers[not_relevant_publishers["possible_to_calculate"] == "No"]
    .groupby("publisher")
    .apply(lambda x: ", ".join(x["rule"].astype(str) + ": " + x["missing_fields"].astype(str)))
    .reset_index()
    .rename(columns={0: "failed rules"})
)

Check the results

In [ ]:
non_relevant_rules